# Calidad del Aire — PM2.5 RMCAB Bogotá
> **Contexto:** `docs/fuentes/calidad_aire.md`  
> **Plan:** `Plan.md` Fase 4 — MVP predictivo  
> **Variable:** PM2.5 (µg/m³) | **Fuente:** RMCAB Bogotá  
> **Objetivo:** Ciclo estadístico completo + comparación de modelos + reporte automático

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means, plot_correlation_heatmap
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.descriptive.temporal import decompose_stl
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_probability
from estadistica_ambiental.predictive.classical import SARIMAXModel
from estadistica_ambiental.predictive.ml import XGBoostModel, RandomForestModel
from estadistica_ambiental.evaluation.metrics import evaluate
from estadistica_ambiental.evaluation.backtesting import walk_forward, compare_backtests
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.reporting.forecast_report import forecast_report
from estadistica_ambiental.config import NORMA_CO, NORMA_OMS

print("Setup OK")

## 1. Datos
> Dataset sintético representativo de RMCAB.  
> Reemplazar con CSV real en `data/raw/` cuando esté disponible.

In [ ]:
np.random.seed(42)
n = 365 * 3
fechas = pd.date_range("2020-01-01", periods=n, freq="D")

trend    = np.linspace(12, 18, n)
seasonal = 6 * np.sin(2 * np.pi * np.arange(n) / 365 - np.pi/2)
noise    = np.random.normal(0, 3, n)
episodes = np.where(np.random.random(n) < 0.03, np.random.uniform(50, 120, n), 0)
pm25     = np.clip(trend + seasonal + noise + episodes, 0, None)

temp    = 13 + 4*np.sin(2*np.pi*np.arange(n)/365) + np.random.normal(0,1,n)
humedad = 75 + 10*np.sin(2*np.pi*np.arange(n)/365 + np.pi) + np.random.normal(0,5,n)
viento  = np.abs(np.random.normal(2, 1, n))

missing_idx = np.random.choice(n, int(n*0.05), replace=False)
pm25[missing_idx] = np.nan

df = pd.DataFrame({"fecha": fechas, "pm25": pm25,
                   "temperatura": temp.round(1),
                   "humedad": humedad.round(1), "viento": viento.round(2)})
print(f"Dataset: {df.shape[0]:,} filas | Faltantes PM2.5: {df.pm25.isna().sum()}")
df.head()

## 2. Validación y EDA

In [ ]:
val = validate(df, date_col="fecha")
print(val.summary())

In [ ]:
quality = assess_quality(df, date_col="fecha")
print(quality.summary())

In [ ]:
report_path = run_eda(df, output="data/output/reports/eda_calidad_aire.html",
                      title="EDA PM2.5 RMCAB Bogotá",
                      date_col="fecha", use_ydata=False)
print(f"Reporte: {report_path}")

## 3. Visualización

In [ ]:
fig = plot_series(df, "fecha", "pm25", title="PM2.5 diario — RMCAB Bogotá")
ax = fig.axes[0]
ax.axhline(NORMA_CO["pm25_24h"],  color="red",    ls="--", lw=1,
           label=f"Norma CO 24h ({NORMA_CO['pm25_24h']} µg/m³)")
ax.axhline(NORMA_OMS["pm25_24h"], color="orange", ls="--", lw=1,
           label=f"Guía OMS ({NORMA_OMS['pm25_24h']} µg/m³)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "pm25", period="month")
plt.show()

## 4. Estadística descriptiva

In [ ]:
summarize(df)

In [ ]:
exc = exceedance_probability(df["pm25"].dropna(), threshold=NORMA_CO["pm25_24h"])
exc_oms = exceedance_probability(df["pm25"].dropna(), threshold=NORMA_OMS["pm25_24h"])
print(f"Norma CO:  {exc['n_exceedances']} días ({exc['pct_exceed']:.1f}%)")
print(f"Guía OMS:  {exc_oms['n_exceedances']} días ({exc_oms['pct_exceed']:.1f}%)")

## 5. Inferencial

In [ ]:
pm25_clean = df.set_index("fecha")["pm25"].dropna()
stationarity_report(pm25_clean)

In [ ]:
mk = mann_kendall(pm25_clean)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.4f} µg/m³/día")

In [ ]:
stl = decompose_stl(pm25_clean, period=365)
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, col, lbl in zip(axes,
    ["observed","trend","seasonal","residual"],
    ["Observado","Tendencia","Estacionalidad","Residuo"]):
    ax.plot(stl[col], lw=1, color="#1a5276")
    ax.set_ylabel(lbl, fontsize=8)
    ax.grid(alpha=0.3)
fig.suptitle("STL — PM2.5", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. Preprocesamiento

In [ ]:
df_imp = impute(df.copy(), cols=["pm25"], method="linear")
print(f"Faltantes antes={df.pm25.isna().sum()} | después={df_imp.pm25.isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_imp.set_index("fecha")["pm25"]
X  = df_imp.set_index("fecha")[["temperatura","humedad","viento"]]

models = {
    "SARIMAX":      SARIMAXModel(order=(1,1,1), seasonal_order=(1,1,1,7)),
    "XGBoost":      XGBoostModel(lags=[1,2,3,7,14]),
    "RandomForest": RandomForestModel(lags=[1,2,3,7,14]),
}

results = {}
for name, model in models.items():
    print(f"Evaluando {name}...", end=" ")
    results[name] = walk_forward(model, ts, horizon=7, n_splits=4)
    print(f"RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
ranking = rank_models(results, domain="air_quality")
ranking[["rmse","mae","r2","score","rank"]]

## 8. Reporte final

In [ ]:
train, test = ts.iloc[:-30], ts.iloc[-30:]
X_tr,  X_te = X.iloc[:-30],  X.iloc[-30:]

preds = {}
for name, model in models.items():
    model.fit(train, X_tr if name=="SARIMAX" else None)
    preds[name] = model.predict(30, X_te if name=="SARIMAX" else None)

mets = {n: evaluate(test.values, p, domain="air_quality") for n, p in preds.items()}
rpt  = forecast_report(test, preds, mets,
    output="data/output/reports/forecast_calidad_aire.html",
    title="Pronóstico PM2.5 RMCAB", variable_name="PM2.5", unit="µg/m³")
print(f"Reporte: {rpt}")

## 9. Conclusiones

- Ver excedencias de norma en sección 4.
- Ver tendencia Mann-Kendall en sección 5.
- Ver mejor modelo en ranking sección 7.

### Datos reales sugeridos
- [RMCAB](http://rmcab.ambientebogota.gov.co/)
- [SIATA](https://siata.gov.co)
- [OpenAQ](https://openaq.org)